In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install facenet-pytorch

In [ ]:
import os
import pickle
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch

from facenet_pytorch import MTCNN
from facenet_pytorch import InceptionResnetV1

[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print("Using Device:", device)

mtcnn = MTCNN(
    image_size=160,
    margin=20,
    device=device
)

facenet = InceptionResnetV1(
    pretrained='vggface2'
).eval().to(device)

Using Device: cuda


  0%|          | 0.00/107M [00:00<?, ?B/s]

In [ ]:
dataset_path = "/content/drive/MyDrive/Faceproject/Dataset/lfw-deepfunneled/lfw-deepfunneled"

people = os.listdir(dataset_path)

print("Total People:", len(people))

Total People: 5749


In [ ]:
people_count = []

for person in os.listdir(dataset_path):

    person_path = os.path.join(dataset_path, person)

    if os.path.isdir(person_path):

        count = len(os.listdir(person_path))

        if count >= 5:

            people_count.append((person, count))

people_count.sort(
    key=lambda x: x[1],
    reverse=True
)

print("People with >=5 images:", len(people_count))

people_count[:20]

People with >=5 images: 423


[('George_W_Bush', 530),
 ('Colin_Powell', 236),
 ('Tony_Blair', 144),
 ('Donald_Rumsfeld', 121),
 ('Gerhard_Schroeder', 109),
 ('Ariel_Sharon', 77),
 ('Hugo_Chavez', 71),
 ('Junichiro_Koizumi', 60),
 ('Jean_Chretien', 55),
 ('John_Ashcroft', 53),
 ('Jacques_Chirac', 52),
 ('Serena_Williams', 52),
 ('Vladimir_Putin', 49),
 ('Luiz_Inacio_Lula_da_Silva', 48),
 ('Gloria_Macapagal_Arroyo', 44),
 ('Arnold_Schwarzenegger', 42),
 ('Jennifer_Capriati', 42),
 ('Lleyton_Hewitt', 41),
 ('Laura_Bush', 41),
 ('Alejandro_Toledo', 39)]

In [ ]:
top_people = [x[0] for x in people_count[:50]]

print(top_people)

['George_W_Bush', 'Colin_Powell', 'Tony_Blair', 'Donald_Rumsfeld', 'Gerhard_Schroeder', 'Ariel_Sharon', 'Hugo_Chavez', 'Junichiro_Koizumi', 'Jean_Chretien', 'John_Ashcroft', 'Jacques_Chirac', 'Serena_Williams', 'Vladimir_Putin', 'Luiz_Inacio_Lula_da_Silva', 'Gloria_Macapagal_Arroyo', 'Arnold_Schwarzenegger', 'Jennifer_Capriati', 'Lleyton_Hewitt', 'Laura_Bush', 'Alejandro_Toledo', 'Hans_Blix', 'Nestor_Kirchner', 'Andre_Agassi', 'Alvaro_Uribe', 'Megawati_Sukarnoputri', 'Tom_Ridge', 'Silvio_Berlusconi', 'Kofi_Annan', 'Roh_Moo-hyun', 'Vicente_Fox', 'David_Beckham', 'John_Negroponte', 'Guillermo_Coria', 'Recep_Tayyip_Erdogan', 'Bill_Clinton', 'Mahmoud_Abbas', 'Juan_Carlos_Ferrero', 'Jack_Straw', 'Ricardo_Lagos', 'Gray_Davis', 'Rudolph_Giuliani', 'Tom_Daschle', 'Atal_Bihari_Vajpayee', 'Jeremy_Greenstock', 'Winona_Ryder', 'Jose_Maria_Aznar', 'Saddam_Hussein', 'Tiger_Woods', 'George_Robertson', 'Hamid_Karzai']


In [ ]:
embeddings_database = {}

for person in tqdm(top_people):

    person_folder = os.path.join(
        dataset_path,
        person
    )

    person_embeddings = []

    for image_name in os.listdir(person_folder):

        image_path = os.path.join(
            person_folder,
            image_name
        )

        try:

            img = Image.open(
                image_path
            ).convert('RGB')

            face = mtcnn(img)

            if face is None:
                continue

            face = face.unsqueeze(0).to(device)

            embedding = facenet(face)

            embedding = embedding.detach().cpu().numpy()[0]

            person_embeddings.append(
                embedding
            )

        except:
            continue

    if len(person_embeddings) > 0:

      embeddings_database[person] = person_embeddings

print(
    "Identities Stored:",
    len(embeddings_database)
)

100%|██████████| 50/50 [24:19<00:00, 29.19s/it]

Identities Stored: 50


In [ ]:
save_path = "/content/drive/MyDrive/Faceproject/models/embeddings.pkl"

with open(save_path, "wb") as f:
    pickle.dump(
        embeddings_database,
        f
    )

print("Saved:", save_path)

Saved: /content/drive/MyDrive/Faceproject/models/embeddings.pkl


In [ ]:
with open(
    "/content/drive/MyDrive/Faceproject/models/embeddings.pkl",
    "rb"
) as f:

    embeddings_database = pickle.load(f)

print(
    len(embeddings_database)
)

50


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving George_W_Bush_0530.jpg to George_W_Bush_0530.jpg


In [ ]:
def recognize_face(image_path):

    img = Image.open(image_path).convert('RGB')

    face = mtcnn(img)

    if face is None:
        return "No Face", 0

    face = face.unsqueeze(0).to(device)

    embedding = facenet(face)

    embedding = embedding.detach().cpu().numpy()[0]

    min_distance = float('inf')
    predicted_person = None

    for person, embeddings in embeddings_database.items():

        for stored_embedding in embeddings:

            distance = np.linalg.norm(
                embedding - stored_embedding
            )

            if distance < min_distance:

                min_distance = distance
                predicted_person = person

    confidence = max(
        0,
        100 - min_distance * 20
    )

    return predicted_person, confidence

In [ ]:
def recognize_top5(image_path):

    img = Image.open(image_path).convert('RGB')

    face = mtcnn(img)

    face = face.unsqueeze(0).to(device)

    embedding = facenet(face)

    embedding = embedding.detach().cpu().numpy()[0]

    distances = []

    for person, stored_embedding in embeddings_database.items():

        distance = np.linalg.norm(
            embedding - stored_embedding
        )

        distances.append((person, distance))

    distances.sort(key=lambda x: x[1])

    return distances[:5]

In [ ]:
image_path = list(uploaded.keys())[0]

person, confidence = recognize_face(
    image_path
)

print("Predicted Person:", person)
print("Confidence:", confidence)

Predicted Person: George_W_Bush
Confidence: 100.0
